# 🎵 Idle 상태 컬럼 분류 모델 (Waveform + Mel Spectrogram with Masking)

이 노트북에서는 **Idle 상태**의 오디오 데이터만 사용하여 컬럼(문제)을 분류하는 모델을 구축합니다.
- Waveform과 Mel Spectrogram을 모두 사용
- Mel Spectrogram 마스킹을 통한 중요 영역 강조
- 마스크 시각화 포함
- 앙상블 모델로 예측

## 📋 목차
1. **데이터 로드**: Idle 상태 데이터만 필터링
2. **Waveform 및 Mel Spectrogram 분석**
3. **마스크 생성 및 시각화**
4. **모델 정의**: Waveform CNN + Masked Mel Spectrogram CNN
5. **앙상블 모델**: Vote 방식 결합
6. **학습 및 평가**

In [ ]:
# ============================================================
# 필수 라이브러리 임포트
# ============================================================

import os
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from tqdm import tqdm
from collections import Counter, defaultdict
import librosa
import librosa.display

# PyTorch
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

# 머신러닝
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

# 공통 유틸리티
from utils import setup_plotting, get_data_dir, get_state_mapping, get_state_names

# 프로젝트 모듈
from app.ml.features.extractor import AudioFeatureExtractor, AudioConfig

# 시각화 설정
setup_plotting()

# 디바이스 설정
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"✅ 라이브러리 로드 완료!")
print(f"🖥️ Device: {device}")

---
## 1. Idle 상태 데이터 로드

In [ ]:
# ============================================================
# Idle 상태 데이터만 로드 (Combined 제외)
# ============================================================

data_dir = get_data_dir()

# 오디오 파일 수집 (Idle 상태만, combined 제외)
all_files = []
all_labels = []

# 디렉토리 이름 확인 (idle state 또는 idle)
idle_dir = None
for state_dir in sorted(data_dir.iterdir()):
    if state_dir.is_dir() and 'idle' in state_dir.name.lower():
        idle_dir = state_dir
        break

if idle_dir is not None and idle_dir.exists() and idle_dir.is_dir():
    print(f"✅ Idle 상태 디렉토리 찾음: {idle_dir.name}")
    for problem_dir in sorted(idle_dir.iterdir()):
        if not problem_dir.is_dir():
            continue
        
        problem_name = problem_dir.name
        
        # Combined 클래스 제외
        if 'combined' in problem_name.lower():
            continue
        
        # 하위 폴더 확인 (combined 제외)
        has_combined = False
        try:
            for sub_dir in problem_dir.iterdir():
                if sub_dir.is_dir() and 'combined' in sub_dir.name.lower():
                    has_combined = True
                    break
        except:
            pass
        
        if has_combined:
            continue
        
        # WAV 파일 수집
        wav_files = list(problem_dir.glob('*.wav'))
        
        for wav_file in wav_files:
            all_files.append(wav_file)
            all_labels.append(problem_name)
        
        # 하위 폴더의 WAV 파일 수집 (combined 제외)
        try:
            for sub_dir in problem_dir.iterdir():
                if sub_dir.is_dir() and 'combined' not in sub_dir.name.lower():
                    sub_wav_files = list(sub_dir.glob('*.wav'))
                    for wav_file in sub_wav_files:
                        all_files.append(wav_file)
                        all_labels.append(f"{problem_name}/{sub_dir.name}")
        except:
            pass
else:
    print(f"⚠️ Idle 상태 디렉토리를 찾을 수 없습니다!")
    print(f"   확인된 디렉토리:")
    for d in sorted(data_dir.iterdir()):
        if d.is_dir():
            print(f"     - {d.name}")

print(f"\n📊 Idle 상태 데이터: {len(all_files)}개")
if len(all_files) > 0:
    print(f"📊 컬럼별 분포:")
    # 컬럼별 분포 확인
    column_counts = Counter(all_labels)
    for column, count in sorted(column_counts.items()):
        print(f"  {column}: {count}개")
else:
    print("⚠️ 데이터가 없습니다. 디렉토리 경로를 확인해주세요.")

---
## 2. Waveform 및 Mel Spectrogram 추출 및 분석

In [ ]:
# ============================================================
# 오디오 특징 추출 설정
# ============================================================

# ============================================================
# 실제 오디오 길이 확인
# ============================================================

print("📊 실제 오디오 길이 확인 중...")
durations = []
for file_path in tqdm(all_files[:100] if len(all_files) > 100 else all_files, desc="길이 확인"):
    try:
        y, sr = librosa.load(str(file_path), sr=None)
        duration = len(y) / sr
        durations.append(duration)
    except:
        continue

if len(durations) > 0:
    durations = np.array(durations)
    print(f"\n📈 오디오 길이 통계:")
    print(f"  • 평균 길이: {np.mean(durations):.2f}초")
    print(f"  • 최소 길이: {np.min(durations):.2f}초")
    print(f"  • 최대 길이: {np.max(durations):.2f}초")
    print(f"  • 표준편차: {np.std(durations):.2f}초")
    
    # duration 설정 (최대 길이의 약 1.2배, 최소 2.0초)
    max_duration = max(np.max(durations) * 1.2, 2.0)
    recommended_duration = min(max_duration, 2.0)  # 최대 2.0초
    print(f"\n💡 권장 duration: {recommended_duration:.1f}초")
else:
    recommended_duration = 2.0
    print(f"⚠️ 오디오 길이를 확인할 수 없어 기본값 사용: {recommended_duration}초")

audio_config = AudioConfig(
    sample_rate=22050,
    duration=recommended_duration,
    n_mels=128,
    n_fft=2048,
    hop_length=512
)

feature_extractor = AudioFeatureExtractor(audio_config)

print(f"\n🔄 Waveform 및 Mel Spectrogram 추출 중... (duration: {audio_config.duration}초)")

# Waveform 및 Mel Spectrogram 추출
X_waveform = []
X_mel = []
y_columns = []

# 실제 오디오 길이 저장 (시각화용)
actual_durations = []

for file_path, label in tqdm(zip(all_files, all_labels), total=len(all_files)):
    try:
        # 오디오 로드
        y, sr = librosa.load(str(file_path), sr=audio_config.sample_rate)
        
        # 실제 오디오 길이 저장
        actual_duration = len(y) / sr
        actual_durations.append(actual_duration)
        
        # Waveform 추출 (패딩 포함)
        target_length = int(audio_config.sample_rate * audio_config.duration)
        if len(y) < target_length:
            y_padded = np.pad(y, (0, target_length - len(y)), mode='constant')
        elif len(y) > target_length:
            y_padded = y[:target_length]
        else:
            y_padded = y
        
        X_waveform.append(y_padded)
        
                # Mel Spectrogram 추출 (원본 오디오 사용, 패딩 전)
        # librosa.feature.melspectrogram을 사용하여 Mel Spectrogram 추출
        # - n_mels: Mel 필터뱅크 수 (128개)
        # - n_fft: FFT 윈도우 크기 (2048)
        # - hop_length: 윈도우 이동 간격 (512)
        mel_spec = feature_extractor.extract_mel_spectrogram(y, sr, to_db=False)
        # dB 스케일로 변환 (시각화 및 학습에 유리)
        mel_spec_db = librosa.power_to_db(mel_spec, ref=np.max)
        X_mel.append(mel_spec_db)
        
        # 컬럼 레이블
        y_columns.append(label)
        
    except Exception as e:
        print(f"⚠️ 오류 발생: {file_path} - {e}")
        continue

X_waveform = np.array(X_waveform)
X_mel = np.array(X_mel)
y_columns = np.array(y_columns)
actual_durations = np.array(actual_durations)

print(f"\n✅ 추출 완료!")
print(f"  Waveform shape: {X_waveform.shape}")
print(f"  Mel Spectrogram shape: {X_mel.shape}")
print(f"  컬럼 레이블 shape: {y_columns.shape}")
print(f"  실제 오디오 평균 길이: {np.mean(actual_durations):.2f}초")

# Mel Spectrogram 특징 상세 정보
if len(X_mel) > 0:
    print(f"\n📊 Mel Spectrogram 특징 정보:")
    print(f"  • 주파수 차원 (Mel bins): {X_mel[0].shape[0]}")
    print(f"  • 시간 차원 (frames): {X_mel[0].shape[1]}")
    print(f"  • 샘플링 레이트: {audio_config.sample_rate} Hz")
    print(f"  • FFT 크기: {audio_config.n_fft}")
    print(f"  • Hop length: {audio_config.hop_length}")
    print(f"  • 시간 해상도: {audio_config.hop_length / audio_config.sample_rate * 1000:.2f} ms/frame")
    print(f"  • 주파수 해상도: {audio_config.sample_rate / audio_config.n_fft:.2f} Hz/bin")

In [ ]:
# ============================================================
# Waveform 분석 및 시각화
# ============================================================

# 컬럼별로 샘플 선택 (시각화용)
unique_columns = sorted(np.unique(y_columns))
sample_indices = {}

for column in unique_columns[:5]:  # 처음 5개 컬럼만
    indices = np.where(y_columns == column)[0]
    if len(indices) > 0:
        sample_indices[column] = indices[0]

# Waveform 시각화
fig, axes = plt.subplots(len(sample_indices), 1, figsize=(14, 3 * len(sample_indices)))
if len(sample_indices) == 1:
    axes = [axes]

for idx, (column, sample_idx) in enumerate(sample_indices.items()):
    waveform = X_waveform[sample_idx]
    actual_duration = actual_durations[sample_idx]
    
    # 실제 오디오 길이만 표시
    actual_samples = int(audio_config.sample_rate * actual_duration)
    waveform_actual = waveform[:actual_samples]
    time_axis = np.linspace(0, actual_duration, len(waveform_actual))
    
    axes[idx].plot(time_axis, waveform_actual, linewidth=0.8, color='blue')
    
    # 패딩 영역 표시 (있는 경우)
    if actual_duration < audio_config.duration:
        padding_start = actual_duration
        padding_samples = len(waveform) - actual_samples
        if padding_samples > 0:
            padding_waveform = waveform[actual_samples:]
            padding_time = np.linspace(padding_start, audio_config.duration, len(padding_waveform))
            axes[idx].plot(padding_time, padding_waveform, linewidth=0.5, color='gray', alpha=0.3, label='Padding')
            axes[idx].axvline(x=actual_duration, color='red', linestyle='--', linewidth=1, label='Actual End')
    
    axes[idx].set_title(f'Waveform - {column} (실제 길이: {actual_duration:.2f}초)', fontsize=12, fontweight='bold')
    axes[idx].set_xlabel('Time (seconds)')
    axes[idx].set_ylabel('Amplitude')
    axes[idx].set_xlim(0, min(actual_duration * 1.1, audio_config.duration))
    axes[idx].grid(True, alpha=0.3)
    if actual_duration < audio_config.duration:
        axes[idx].legend()

plt.tight_layout()
plt.show()

print("✅ Waveform 시각화 완료!")

In [ ]:
# ============================================================
# Mel Spectrogram 분석 및 시각화
# ============================================================

# Mel Spectrogram 시각화
fig, axes = plt.subplots(len(sample_indices), 1, figsize=(14, 4 * len(sample_indices)))
if len(sample_indices) == 1:
    axes = [axes]

for idx, (column, sample_idx) in enumerate(sample_indices.items()):
    mel_spec_db = X_mel[sample_idx]  # 이미 dB 스케일로 변환됨
    actual_duration = actual_durations[sample_idx]
    
    # 시간 축 계산 (초 단위)
    hop_length = audio_config.hop_length
    time_frames = mel_spec_db.shape[1]
    time_axis_sec = np.linspace(0, actual_duration, time_frames)
    
    # 실제 오디오 길이에 맞게 표시
    extent = [0, actual_duration, 0, audio_config.n_mels]
    im = axes[idx].imshow(mel_spec_db, aspect='auto', origin='lower', cmap='viridis', extent=extent)
    axes[idx].set_title(f'Mel Spectrogram - {column} (실제 길이: {actual_duration:.2f}초)', fontsize=12, fontweight='bold')
    axes[idx].set_xlabel('Time (seconds)')
    axes[idx].set_ylabel('Mel Frequency Bins')
    axes[idx].set_xlim(0, actual_duration)
    plt.colorbar(im, ax=axes[idx], label='dB')

plt.tight_layout()
plt.show()

print("✅ Mel Spectrogram 시각화 완료!")

---
## 3. 마스크 생성 및 시각화

In [ ]:
# ============================================================
# 컬럼별 평균 Mel Spectrogram 계산
# ============================================================

column_means = {}
for column in unique_columns:
    column_indices = np.where(y_columns == column)[0]
    if len(column_indices) > 0:
        column_means[column] = np.mean(X_mel[column_indices], axis=0)

print(f"✅ 컬럼별 평균 Mel Spectrogram 계산 완료! ({len(column_means)}개 컬럼)")

In [ ]:
# ============================================================
# 중요 영역 마스크 생성 (컬럼별 차이가 큰 영역)
# ============================================================

# 중요 영역 마스크 생성
importance_mask = np.zeros_like(X_mel[0])

columns_list = list(column_means.keys())
for i in range(len(columns_list)):
    for j in range(i + 1, len(columns_list)):
        diff = np.abs(column_means[columns_list[i]] - column_means[columns_list[j]])
        importance_mask = np.maximum(importance_mask, diff)

# 정규화
if importance_mask.max() > importance_mask.min():
    importance_mask = (importance_mask - importance_mask.min()) / (importance_mask.max() - importance_mask.min())
else:
    importance_mask = np.ones_like(importance_mask) * 0.5

# 마스크를 텐서로 변환
importance_mask_tensor = torch.FloatTensor(importance_mask).unsqueeze(0).to(device)

print(f"✅ 중요 영역 마스크 생성 완료!")
print(f"  마스크 shape: {importance_mask.shape}")
print(f"  마스크 범위: [{importance_mask.min():.3f}, {importance_mask.max():.3f}]")
print(f"  마스크 평균값: {importance_mask.mean():.3f}")
print(f"  마스크 중앙값: {np.median(importance_mask):.3f}")
print(f"  마스크 표준편차: {importance_mask.std():.3f}")

# 마스크 분포 확인 (0.5 이상인 영역 비율)
high_importance_ratio = (importance_mask >= 0.5).sum() / importance_mask.size
print(f"  높은 중요도 영역 (≥0.5) 비율: {high_importance_ratio * 100:.1f}%")

# 마스크 히스토그램 시각화
fig, ax = plt.subplots(1, 1, figsize=(10, 6))
ax.hist(importance_mask.flatten(), bins=50, edgecolor='black', alpha=0.7)
ax.axvline(importance_mask.mean(), color='red', linestyle='--', label=f'평균: {importance_mask.mean():.3f}')
ax.axvline(np.median(importance_mask), color='green', linestyle='--', label=f'중앙값: {np.median(importance_mask):.3f}')
ax.set_xlabel('마스크 값 (중요도)', fontsize=12)
ax.set_ylabel('빈도', fontsize=12)
ax.set_title('중요 영역 마스크 분포', fontsize=14, fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"\n💡 마스크 해석:")
print(f"  • 마스크 값이 0에 가까울수록: 컬럼 간 차이가 작은 영역 (덜 중요)")
print(f"  • 마스크 값이 1에 가까울수록: 컬럼 간 차이가 큰 영역 (중요)")
print(f"  • 정규화로 인해 범위는 항상 [0, 1]입니다.")

In [ ]:
# ============================================================
# 마스크 시각화
# ============================================================

fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# 1. 원본 Mel Spectrogram (예시)
sample_idx = 0
mel_spec_db = X_mel[sample_idx]  # 이미 dB 스케일로 변환됨
im1 = axes[0, 0].imshow(mel_spec_db, aspect='auto', origin='lower', cmap='viridis')
axes[0, 0].set_title('Original Mel Spectrogram (Example)', fontsize=12, fontweight='bold')
axes[0, 0].set_xlabel('Time (frames)')
axes[0, 0].set_ylabel('Mel Frequency Bins')
plt.colorbar(im1, ax=axes[0, 0], label='dB')

# 2. 중요 영역 마스크
im2 = axes[0, 1].imshow(importance_mask, aspect='auto', origin='lower', cmap='hot')
axes[0, 1].set_title('Importance Mask (High = Important)', fontsize=12, fontweight='bold')
axes[0, 1].set_xlabel('Time (frames)')
axes[0, 1].set_ylabel('Mel Frequency Bins')
plt.colorbar(im2, ax=axes[0, 1], label='Importance')

# 3. 마스크 적용된 Mel Spectrogram
# 원본을 power 스케일로 변환 후 마스크 적용
mel_spec_power = librosa.db_to_power(mel_spec_db)
masked_mel_power = mel_spec_power * (1 + importance_mask)
masked_mel_db = librosa.power_to_db(masked_mel_power, ref=np.max)
im3 = axes[1, 0].imshow(masked_mel_db, aspect='auto', origin='lower', cmap='viridis')
axes[1, 0].set_title('Masked Mel Spectrogram (Example)', fontsize=12, fontweight='bold')
axes[1, 0].set_xlabel('Time (frames)')
axes[1, 0].set_ylabel('Mel Frequency Bins')
plt.colorbar(im3, ax=axes[1, 0], label='dB')

# 4. 차이 시각화
diff = masked_mel_db - mel_spec_db
im4 = axes[1, 1].imshow(diff, aspect='auto', origin='lower', cmap='RdBu_r')
axes[1, 1].set_title('Difference (Masked - Original)', fontsize=12, fontweight='bold')
axes[1, 1].set_xlabel('Time (frames)')
axes[1, 1].set_ylabel('Mel Frequency Bins')
plt.colorbar(im4, ax=axes[1, 1], label='dB Difference')

plt.tight_layout()
plt.show()

print("✅ 마스크 시각화 완료!")

In [ ]:
# ============================================================
# 컬럼별 평균 Mel Spectrogram 비교 시각화
# ============================================================

# 처음 4개 컬럼만 비교
columns_to_compare = list(column_means.keys())[:4]

fig, axes = plt.subplots(2, 2, figsize=(16, 12))
axes = axes.flatten()

for idx, column in enumerate(columns_to_compare):
    if column in column_means:
        # column_means는 이미 dB 스케일이므로 그대로 사용
        mel_mean_db = column_means[column]
        im = axes[idx].imshow(mel_mean_db, aspect='auto', origin='lower', cmap='viridis')
        axes[idx].set_title(f'Average Mel Spectrogram - {column}', fontsize=11, fontweight='bold')
        axes[idx].set_xlabel('Time (frames)')
        axes[idx].set_ylabel('Mel Frequency Bins')
        plt.colorbar(im, ax=axes[idx], label='dB')

plt.tight_layout()
plt.show()

print("✅ 컬럼별 평균 Mel Spectrogram 비교 완료!")

---
## 4. 모델 정의

In [ ]:
# ============================================================
# Waveform 1D CNN 모델
# ============================================================

class WaveformCNN1D(nn.Module):
    """Waveform을 입력으로 받는 1D CNN 모델"""
    
    def __init__(
        self,
        num_classes: int,
        input_length: int = 110250,  # 5초 @ 22050 Hz
        base_channels: int = 64,
        dropout: float = 0.3
    ):
        super().__init__()
        
        self.num_classes = num_classes
        
        # 1D Convolutional layers
        self.conv1 = nn.Sequential(
            nn.Conv1d(1, base_channels, kernel_size=7, stride=2, padding=3),
            nn.BatchNorm1d(base_channels),
            nn.ReLU(inplace=True),
            nn.MaxPool1d(kernel_size=2, stride=2),
            nn.Dropout(dropout / 2)
        )
        
        self.conv2 = nn.Sequential(
            nn.Conv1d(base_channels, base_channels * 2, kernel_size=5, stride=2, padding=2),
            nn.BatchNorm1d(base_channels * 2),
            nn.ReLU(inplace=True),
            nn.MaxPool1d(kernel_size=2, stride=2),
            nn.Dropout(dropout / 2)
        )
        
        self.conv3 = nn.Sequential(
            nn.Conv1d(base_channels * 2, base_channels * 4, kernel_size=3, stride=2, padding=1),
            nn.BatchNorm1d(base_channels * 4),
            nn.ReLU(inplace=True),
            nn.MaxPool1d(kernel_size=2, stride=2),
            nn.Dropout(dropout / 2)
        )
        
        # Global Average Pooling
        self.global_pool = nn.AdaptiveAvgPool1d(1)
        
        # Fully connected layers
        self.fc1 = nn.Sequential(
            nn.Linear(base_channels * 4, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout)
        )
        
        self.fc2 = nn.Sequential(
            nn.Linear(256, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout)
        )
        
        self.fc_out = nn.Linear(128, num_classes)
        
    def forward(self, x):
        # x: (batch, 1, length)
        x = self.conv1(x)
        x = self.conv2(x)
        x = self.conv3(x)
        x = self.global_pool(x)
        x = x.view(x.size(0), -1)
        x = self.fc1(x)
        x = self.fc2(x)
        x = self.fc_out(x)
        return x

print("✅ WaveformCNN1D 모델 정의 완료!")

In [ ]:
# ============================================================
# 마스킹 기반 Mel Spectrogram CNN 모델
# ============================================================

class MaskedSpatialAttention(nn.Module):
    """마스킹 기반 Spatial Attention"""
    
    def __init__(self, importance_mask: torch.Tensor, learnable: bool = True):
        super().__init__()
        self.importance_mask = nn.Parameter(importance_mask, requires_grad=learnable)
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: (batch, channels, freq, time)
        # 마스크를 입력 shape에 맞게 조정
        mask = self.importance_mask.expand(x.size(0), -1, -1)
        mask = mask.unsqueeze(1)  # (batch, 1, freq, time)
        
        # 마스크 적용 (중요 영역 강조)
        x = x * (1 + mask)
        return x


class MaskedCNN(nn.Module):
    """마스킹 기반 Mel Spectrogram CNN 모델"""
    
    def __init__(self, num_classes: int, importance_mask: torch.Tensor, 
                 in_channels: int = 1, base_channels: int = 32, dropout: float = 0.3):
        super().__init__()
        
        self.num_classes = num_classes
        
        # 마스킹 기반 Spatial Attention
        self.masked_attention = MaskedSpatialAttention(importance_mask, learnable=True)
        
        # Convolutional layers
        self.conv1 = nn.Sequential(
            nn.Conv2d(in_channels, base_channels, 3, padding=1),
            nn.BatchNorm2d(base_channels),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
            nn.Dropout2d(dropout)
        )
        
        self.conv2 = nn.Sequential(
            nn.Conv2d(base_channels, base_channels * 2, 3, padding=1),
            nn.BatchNorm2d(base_channels * 2),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
            nn.Dropout2d(dropout)
        )
        
        self.conv3 = nn.Sequential(
            nn.Conv2d(base_channels * 2, base_channels * 4, 3, padding=1),
            nn.BatchNorm2d(base_channels * 4),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
            nn.Dropout2d(dropout)
        )
        
        self.conv4 = nn.Sequential(
            nn.Conv2d(base_channels * 4, base_channels * 8, 3, padding=1),
            nn.BatchNorm2d(base_channels * 8),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
            nn.Dropout2d(dropout)
        )
        
        # Global Average Pooling
        self.global_pool = nn.AdaptiveAvgPool2d((1, 1))
        
        # Fully connected layers
        self.fc1 = nn.Sequential(
            nn.Linear(base_channels * 8, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout)
        )
        
        self.fc2 = nn.Sequential(
            nn.Linear(256, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout)
        )
        
        self.fc_out = nn.Linear(128, num_classes)
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # 마스킹 기반 Attention 적용
        x = self.masked_attention(x)
        
        # Convolutional blocks
        x = self.conv1(x)
        x = self.conv2(x)
        x = self.conv3(x)
        x = self.conv4(x)
        
        # Global Average Pooling
        x = self.global_pool(x)
        x = x.view(x.size(0), -1)
        
        # Fully connected layers
        x = self.fc1(x)
        x = self.fc2(x)
        x = self.fc_out(x)
        
        return x

print("✅ MaskedCNN 모델 정의 완료!")

---
## 5. 앙상블 모델 (Vote 방식)

In [ ]:
# ============================================================
# 앙상블 모델 클래스 (Vote 방식)
# ============================================================

class EnsembleVoteModel(nn.Module):
    """
    Waveform과 Mel Spectrogram 모델을 Vote 방식으로 결합
    """
    
    def __init__(
        self,
        waveform_model: nn.Module,
        spectrogram_model: nn.Module,
        vote_method: str = 'soft'  # 'hard' or 'soft'
    ):
        super().__init__()
        
        self.waveform_model = waveform_model
        self.spectrogram_model = spectrogram_model
        self.vote_method = vote_method
    
    def forward(self, waveform_input, spectrogram_input):
        """
        두 모델의 예측을 결합
        
        Args:
            waveform_input: Waveform 텐서 (batch, 1, length)
            spectrogram_input: Mel Spectrogram 텐서 (batch, 1, freq, time)
        
        Returns:
            ensemble_output: 앙상블 예측 결과 (batch, num_classes) - logits
        """
        # Waveform 모델 예측
        waveform_output = self.waveform_model(waveform_input)
        
        # Mel Spectrogram 모델 예측
        spec_output = self.spectrogram_model(spectrogram_input)
        
        # Vote 방식에 따라 결합
        if self.vote_method == 'hard':
            # Hard Voting: 다수결
            waveform_pred = waveform_output.argmax(dim=1)
            spec_pred = spec_output.argmax(dim=1)
            
            ensemble_pred = torch.stack([waveform_pred, spec_pred], dim=1)
            ensemble_pred = torch.mode(ensemble_pred, dim=1)[0]
            
            # One-hot으로 변환 후 logits로 변환
            ensemble_output = F.one_hot(ensemble_pred, num_classes=waveform_output.size(1)).float()
            # logits로 변환 (큰 값 사용)
            ensemble_output = ensemble_output * 10.0 - 5.0
        
        else:  # soft voting
            # Soft Voting: logits 평균 (학습 시 사용)
            ensemble_output = (waveform_output + spec_output) / 2
        
        return ensemble_output

print("✅ EnsembleVoteModel 클래스 정의 완료!")

---
## 6. 데이터셋 클래스

In [ ]:
# ============================================================
# 데이터셋 클래스
# ============================================================

class ColumnClassificationDataset(Dataset):
    """Idle 상태 컬럼 분류용 데이터셋"""
    
    def __init__(self, X_waveform, X_mel, y_columns, column_to_idx):
        self.X_waveform = X_waveform
        self.X_mel = X_mel
        self.y_columns = y_columns
        self.column_to_idx = column_to_idx
        
        # 레이블을 인덱스로 변환
        self.y_indices = np.array([column_to_idx[column] for column in y_columns])
    
    def __len__(self):
        return len(self.X_waveform)
    
    def __getitem__(self, idx):
        waveform = torch.FloatTensor(self.X_waveform[idx]).unsqueeze(0)  # (1, length)
        mel_spec = torch.FloatTensor(self.X_mel[idx]).unsqueeze(0)  # (1, freq, time)
        label = torch.LongTensor([self.y_indices[idx]])[0]
        
        return waveform, mel_spec, label

print("✅ ColumnClassificationDataset 클래스 정의 완료!")

---
## 7. 데이터 분할 및 준비

In [ ]:
# ============================================================
# 컬럼 레이블 매핑
# ============================================================

unique_columns = sorted(np.unique(y_columns))
column_to_idx = {column: idx for idx, column in enumerate(unique_columns)}
idx_to_column = {idx: column for column, idx in column_to_idx.items()}
num_classes = len(unique_columns)

print(f"📊 컬럼 클래스:")
for column, idx in column_to_idx.items():
    count = np.sum(y_columns == column)
    print(f"  {column}: {count}개 (인덱스: {idx})")

print(f"\n총 클래스 수: {num_classes}개")

In [ ]:
# ============================================================
# Train/Val/Test 분할
# ============================================================

# 먼저 Train/Test 분할 (80:20)
X_waveform_train, X_waveform_test, X_mel_train, X_mel_test, y_train, y_test = train_test_split(
    X_waveform, X_mel, y_columns, test_size=0.2, random_state=42, stratify=y_columns
)

# Train을 다시 Train/Val 분할 (80:20)
X_waveform_train, X_waveform_val, X_mel_train, X_mel_val, y_train, y_val = train_test_split(
    X_waveform_train, X_mel_train, y_train, test_size=0.2, random_state=42, stratify=y_train
)

print(f"📊 데이터 분할:")
print(f"  Train: {len(X_waveform_train)}개")
print(f"  Val: {len(X_waveform_val)}개")
print(f"  Test: {len(X_waveform_test)}개")

In [ ]:
# ============================================================
# 데이터셋 및 DataLoader 생성
# ============================================================

train_dataset = ColumnClassificationDataset(X_waveform_train, X_mel_train, y_train, column_to_idx)
val_dataset = ColumnClassificationDataset(X_waveform_val, X_mel_val, y_val, column_to_idx)
test_dataset = ColumnClassificationDataset(X_waveform_test, X_mel_test, y_test, column_to_idx)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

print("✅ 데이터셋 및 DataLoader 생성 완료!")

---
## 8. 모델 생성 및 학습

In [ ]:
# ============================================================
# 개별 모델 생성
# ============================================================

# Waveform 모델
# Waveform 모델 (실제 입력 길이 사용)
waveform_input_length = X_waveform.shape[1]
waveform_model = WaveformCNN1D(
    num_classes=num_classes,
    input_length=waveform_input_length,
    base_channels=64,
    dropout=0.3
).to(device)

# Mel Spectrogram 모델
mel_model = MaskedCNN(
    num_classes=num_classes,
    importance_mask=importance_mask_tensor,
    in_channels=1,
    base_channels=32,
    dropout=0.3
).to(device)

# 앙상블 모델
ensemble_model = EnsembleVoteModel(
    waveform_model=waveform_model,
    spectrogram_model=mel_model,
    vote_method='soft'
).to(device)

print("✅ 모델 생성 완료!")
print(f"  Waveform 모델 파라미터: {sum(p.numel() for p in waveform_model.parameters()):,}개")
print(f"  Mel Spectrogram 모델 파라미터: {sum(p.numel() for p in mel_model.parameters()):,}개")
print(f"  앙상블 모델 파라미터: {sum(p.numel() for p in ensemble_model.parameters()):,}개")

In [ ]:
# ============================================================
# 학습 함수
# ============================================================

def train_epoch(model, train_loader, criterion, optimizer, device):
    model.train()
    total_loss = 0
    correct = 0
    total = 0
    
    for waveform, mel_spec, labels in train_loader:
        waveform = waveform.to(device)
        mel_spec = mel_spec.to(device)
        labels = labels.to(device)
        
        optimizer.zero_grad()
        
        # 앙상블 모델 예측 (logits 반환)
        outputs = model(waveform, mel_spec)
        
        # CrossEntropyLoss는 logits를 기대함
        loss = criterion(outputs, labels)
        preds = outputs.argmax(dim=1)
        
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        total += labels.size(0)
        correct += (preds == labels).sum().item()
    
    return total_loss / len(train_loader), correct / total


def validate(model, val_loader, criterion, device):
    model.eval()
    total_loss = 0
    correct = 0
    total = 0
    all_preds = []
    all_labels = []
    
    with torch.no_grad():
        for waveform, mel_spec, labels in val_loader:
            waveform = waveform.to(device)
            mel_spec = mel_spec.to(device)
            labels = labels.to(device)
            
            outputs = model(waveform, mel_spec)
            
            # CrossEntropyLoss는 logits를 기대함
            loss = criterion(outputs, labels)
            preds = outputs.argmax(dim=1)
            
            total_loss += loss.item()
            total += labels.size(0)
            correct += (preds == labels).sum().item()
            
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    
    return total_loss / len(val_loader), correct / total, all_preds, all_labels

print("✅ 학습/검증 함수 정의 완료!")

In [19]:
# ============================================================
# 모델 학습
# ============================================================

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(ensemble_model.parameters(), lr=0.001)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=5)

num_epochs = 50
best_val_acc = 0
train_losses = []
val_losses = []
train_accs = []
val_accs = []

print("🚀 앙상블 모델 학습 시작\n")

for epoch in range(num_epochs):
    train_loss, train_acc = train_epoch(ensemble_model, train_loader, criterion, optimizer, device)
    val_loss, val_acc, _, _ = validate(ensemble_model, val_loader, criterion, device)
    
    scheduler.step(val_loss)
    
    train_losses.append(train_loss)
    val_losses.append(val_loss)
    train_accs.append(train_acc)
    val_accs.append(val_acc)
    
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        model_path = Path('checkpoints')
        model_path.mkdir(exist_ok=True)
        torch.save(ensemble_model.state_dict(), model_path / 'best_idle_column_model.pth')
    
    if (epoch + 1) % 5 == 0:
        print(f"Epoch [{epoch+1}/{num_epochs}]")
        print(f"  Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f}")
        print(f"  Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.4f}")
        print()

print("✅ 학습 완료!")

KeyboardInterrupt: 

---
## 9. 학습 곡선 시각화

In [ ]:
# ============================================================
# 학습 곡선 시각화
# ============================================================

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Loss 곡선
axes[0].plot(train_losses, label='Train Loss', linewidth=2)
axes[0].plot(val_losses, label='Val Loss', linewidth=2)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Training and Validation Loss', fontsize=12, fontweight='bold')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Accuracy 곡선
axes[1].plot(train_accs, label='Train Acc', linewidth=2)
axes[1].plot(val_accs, label='Val Acc', linewidth=2)
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].set_title('Training and Validation Accuracy', fontsize=12, fontweight='bold')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("✅ 학습 곡선 시각화 완료!")

---
## 10. 최종 평가

In [ ]:
# ============================================================
# 최종 테스트 평가
# ============================================================

# 최고 모델 로드
model_path = Path('checkpoints') / 'best_idle_column_model.pth'
if model_path.exists():
    ensemble_model.load_state_dict(torch.load(model_path))
    print(f"✅ 모델 로드 완료: {model_path}")
else:
    print(f"⚠️ 모델 파일을 찾을 수 없습니다: {model_path}")

# 테스트 평가
test_loss, test_acc, test_preds, test_labels = validate(
    ensemble_model, test_loader, criterion, device
)

print(f"\n📊 최종 테스트 결과:")
print(f"  Test Loss: {test_loss:.4f}")
print(f"  Test Accuracy: {test_acc:.4f}")

# 분류 리포트
print(f"\n📋 분류 리포트:")
print(classification_report(
    test_labels,
    test_preds,
    target_names=[idx_to_column[i] for i in range(num_classes)]
))

# 혼동 행렬
cm = confusion_matrix(test_labels, test_preds)
plt.figure(figsize=(12, 10))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=[idx_to_column[i] for i in range(num_classes)],
            yticklabels=[idx_to_column[i] for i in range(num_classes)])
plt.title('Confusion Matrix - Idle Column Classification', fontsize=14, fontweight='bold')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()